# DSFB-DSCD scaling + provenance analysis

This notebook replays deterministic DSCD threshold scaling outputs and provenance exports.

Expected files in `RUN_DIR`:
- `threshold_scaling_summary.csv`
- `finite_size_scaling.csv`
- `threshold_curve_N_<N>.csv`
- `graph_events.csv`
- `graph_edges.csv`
- `edge_provenance_<edge_id>.csv` (at least one)

All plots are exported as paper-ready PNGs into the same run directory.


In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

# Set explicitly if needed: RUN_DIR = Path('output-dsfb-dscd/<timestamp>')
RUN_DIR = None

def find_latest_run() -> Path:
    candidates = []
    roots = [
        Path('output-dsfb-dscd'),
        Path.cwd() / 'output-dsfb-dscd',
        Path('/content/output-dsfb-dscd'),
        Path('/content/dsfb/output-dsfb-dscd'),
    ]
    for root in roots:
        if not root.exists():
            continue
        for run_dir in root.iterdir():
            if run_dir.is_dir() and (run_dir / 'threshold_scaling_summary.csv').exists() and (run_dir / 'finite_size_scaling.csv').exists():
                candidates.append(run_dir.resolve())

    if not candidates:
        raise FileNotFoundError('No DSCD scaling run found with threshold_scaling_summary.csv + finite_size_scaling.csv')

    return sorted(candidates)[-1]

RUN_DIR = Path(RUN_DIR).expanduser().resolve() if RUN_DIR is not None else find_latest_run()
print(f'Using RUN_DIR: {RUN_DIR}')
RUN_DIR


In [ ]:
summary_path = RUN_DIR / 'threshold_scaling_summary.csv'
finite_size_path = RUN_DIR / 'finite_size_scaling.csv'
curve_paths = sorted(RUN_DIR.glob('threshold_curve_N_*.csv'))

if not summary_path.exists():
    raise FileNotFoundError(summary_path)
if not finite_size_path.exists():
    raise FileNotFoundError(finite_size_path)
if not curve_paths:
    raise FileNotFoundError('No threshold_curve_N_*.csv files found in RUN_DIR')

summary_df = pd.read_csv(summary_path).sort_values('N').reset_index(drop=True)
finite_size_df = pd.read_csv(finite_size_path)

curve_by_n = {}
for path in curve_paths:
    match = re.search(r'threshold_curve_N_(\d+)\.csv$', path.name)
    if not match:
        continue
    n = int(match.group(1))
    curve_by_n[n] = pd.read_csv(path).sort_values('tau').reset_index(drop=True)

if not curve_by_n:
    raise RuntimeError('Could not parse N from threshold curve filenames')

print('Loaded scaling summary rows:', len(summary_df))
print('Loaded curve files for N:', sorted(curve_by_n.keys()))
display(summary_df)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for n in sorted(curve_by_n):
    df = curve_by_n[n]
    ax.plot(df['tau'], df['expansion_ratio'], linewidth=2, label=f'N={n}')

ax.set_xlabel('tau')
ax.set_ylabel('expansion ratio')
ax.set_title('DSCD threshold curves across finite sizes')
ax.grid(alpha=0.25)
ax.legend()

fig_path = RUN_DIR / 'fig_dscd_threshold_curves.png'
fig.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved {fig_path}')


In [ ]:
def fit_power_law(x, y, label: str):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    if mask.sum() < 2:
        raise ValueError(f'Need >=2 positive points for power-law fit of {label}')

    lx = np.log10(x[mask])
    ly = np.log10(y[mask])
    slope, intercept = np.polyfit(lx, ly, 1)
    pred = slope * lx + intercept
    ss_res = np.sum((ly - pred) ** 2)
    ss_tot = np.sum((ly - np.mean(ly)) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 1.0

    print(f'{label}: exponent={-slope:.6f}, slope={slope:.6f}, R^2={r2:.6f}')
    return slope, intercept, r2

N = summary_df['N'].to_numpy(dtype=float)
width_col = 'width_0_1_to_0_9' if 'width_0_1_to_0_9' in summary_df.columns else 'transition_width'
width = summary_df[width_col].to_numpy(dtype=float)
max_derivative = summary_df['max_derivative'].to_numpy(dtype=float)
inv_max_derivative = 1.0 / np.clip(max_derivative, 1e-12, None)

width_slope, width_intercept, width_r2 = fit_power_law(N, width, 'width ~ N^(-gamma)')
inv_slope, inv_intercept, inv_r2 = fit_power_law(N, inv_max_derivative, '1/max_derivative ~ N^(-beta)')

fig_w, ax_w = plt.subplots(figsize=(7, 5))
ax_w.loglog(N, width, 'o', label='observed width')
fit_width = 10 ** width_intercept * (N ** width_slope)
ax_w.loglog(N, fit_width, '-', label=f'fit slope={width_slope:.3f}, R2={width_r2:.3f}')
ax_w.set_xlabel('N')
ax_w.set_ylabel('width_0_1_to_0_9')
ax_w.set_title('Finite-size scaling of transition width')
ax_w.grid(alpha=0.25)
ax_w.legend()
fig_w_path = RUN_DIR / 'fig_dscd_width_scaling.png'
fig_w.savefig(fig_w_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved {fig_w_path}')

fig_d, ax_d = plt.subplots(figsize=(7, 5))
ax_d.loglog(N, max_derivative, 'o', label='observed max derivative')
md_slope, md_intercept, md_r2 = fit_power_law(N, max_derivative, 'max_derivative ~ N^(s)')
fit_md = 10 ** md_intercept * (N ** md_slope)
ax_d.loglog(N, fit_md, '-', label=f'fit slope={md_slope:.3f}, R2={md_r2:.3f}')
ax_d.set_xlabel('N')
ax_d.set_ylabel('max_derivative')
ax_d.set_title('Transition sharpening via max derivative')
ax_d.grid(alpha=0.25)
ax_d.legend()
fig_d_path = RUN_DIR / 'fig_dscd_max_derivative_scaling.png'
fig_d.savefig(fig_d_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved {fig_d_path}')



In [ ]:
events_df = pd.read_csv(RUN_DIR / 'graph_events.csv')
edges_df = pd.read_csv(RUN_DIR / 'graph_edges.csv')

prov_paths = sorted(RUN_DIR.glob('edge_provenance_*.csv'))
if prov_paths:
    provenance_df = pd.read_csv(prov_paths[0])
    edge_row = provenance_df.iloc[0]
    print(f'Using provenance file: {prov_paths[0].name}')
else:
    if edges_df.empty:
        raise RuntimeError('graph_edges.csv is empty; cannot build provenance figure')
    edge_row = edges_df.iloc[0]
    print('No edge_provenance_*.csv found; using first row from graph_edges.csv')

required = ['observer_id', 'trust_at_creation', 'source_event_id', 'target_event_id']
for column in required:
    if column not in edge_row.index:
        raise KeyError(f'Missing required provenance column: {column}')

display(edge_row.to_frame(name='value'))


In [ ]:
source_id = int(edge_row['source_event_id'])
target_id = int(edge_row['target_event_id'])
observer_id = int(edge_row['observer_id'])
trust_at_creation = float(edge_row['trust_at_creation'])
rewrite_rule_at_source = int(edge_row['rewrite_rule_at_source']) if 'rewrite_rule_at_source' in edge_row.index else -1

if 'source_event_id' in edges_df.columns and 'target_event_id' in edges_df.columns:
    src_col, dst_col = 'source_event_id', 'target_event_id'
else:
    src_col, dst_col = 'from_event_id', 'to_event_id'

graph = nx.DiGraph()
for row in edges_df.itertuples(index=False):
    src = int(getattr(row, src_col))
    dst = int(getattr(row, dst_col))
    graph.add_edge(src, dst)

# Local neighborhood around source + target (two hops each direction).
neighborhood = {source_id, target_id}
frontier = {source_id, target_id}
for _ in range(2):
    nxt = set()
    for node in frontier:
        nxt.update(graph.successors(node))
        nxt.update(graph.predecessors(node))
    neighborhood.update(nxt)
    frontier = nxt

subgraph = graph.subgraph(sorted(neighborhood)).copy()

fig, ax = plt.subplots(figsize=(9, 6))
pos = nx.spring_layout(subgraph, seed=7)
node_colors = []
for node in subgraph.nodes():
    if node == source_id:
        node_colors.append('#d62728')
    elif node == target_id:
        node_colors.append('#ff7f0e')
    else:
        node_colors.append('#1f77b4')

nx.draw_networkx(subgraph, pos=pos, node_size=220, node_color=node_colors, with_labels=True, arrows=True, ax=ax)
ax.set_title('DSCD provenance neighborhood around selected edge')
ax.axis('off')

src_event_rows = events_df[(events_df['event_id'] == source_id) & (events_df['observer_id'] == observer_id)]
residual_state = src_event_rows['residual_state'].iloc[0] if not src_event_rows.empty else 'NA'
rewrite_rule_id = src_event_rows['rewrite_rule_id'].iloc[0] if not src_event_rows.empty else 'NA'

summary_text = '\n'.join([
    f'observer_id: {observer_id}',
    f'trust_at_creation: {trust_at_creation:.6f}',
    f'residual_state: {residual_state}',
    f'rewrite_rule_at_source: {rewrite_rule_at_source}',
    f'rewrite_rule_id (event row): {rewrite_rule_id}',
])
ax.text(1.02, 0.98, summary_text, transform=ax.transAxes, va='top', fontsize=10, bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

fig_path = RUN_DIR / 'fig_dscd_provenance_neighborhood.png'
fig.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved {fig_path}')


In [ ]:
exported = sorted(RUN_DIR.glob('fig_dscd_*.png'))
print('Exported figure files:')
for path in exported:
    print(' -', path.name)
